<a href="https://colab.research.google.com/github/Saransh-Basu-01/50-Ml-Projects-to-understand-LLMs/blob/main/Project_32_Patching_Hidden_states_in_Indirect_object_identification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

from scipy.optimize import curve_fit

from tqdm import tqdm

from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch

# Part_1:The IOI Task

## Task_1:
- Import GPT-2-xl and move it to the GPU. Tokenize the following
two sentences (excluding the numbers):
1. When Mike and Emma went to the cafe, Mike gave a
coffee to
2. When Mike and Emma went to the cafe, Emma gave a
coffee to
Find the token indices for “ Mike” and “ Emma” (including the
preceding spaces).
Push both token sequences through the model and output the
hidden states. You can do one forward pass with batched input,
although I found it more convenient for subsequent analyses to have
two separate runs with separate output variable names

In [2]:

device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = GPT2LMHeadModel.from_pretrained('gpt2-xl').to(device)
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

n_layers = model.config.n_layer
model.eval()

config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 6.43GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1600)
    (wpe): Embedding(1024, 1600)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-47): 48 x GPT2Block(
        (ln_1): LayerNorm((1600,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=4800, nx=1600)
          (c_proj): Conv1D(nf=1600, nx=1600)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1600,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=6400, nx=1600)
          (c_proj): Conv1D(nf=1600, nx=6400)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1600,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=1600, out_features=50257, bias=False)
)

In [3]:
n_layers

48

In [4]:

text_ME = 'When Mike and Emma went to the cafe, Mike gave a coffee to'
text_EM = 'When Mike and Emma went to the cafe, Emma gave a coffee to'

target_M = tokenizer.encode(' Mike')[0]
target_E = tokenizer.encode(' Emma')[0]

tokensME = tokenizer.encode(text_ME,return_tensors='pt').to(device)
tokensEM = tokenizer.encode(text_EM,return_tensors='pt').to(device)


In [5]:
target_M

4995

In [6]:
tokensEM

tensor([[ 2215,  4995,   290, 18966,  1816,   284,   262, 26725,    11, 18966,
          2921,   257,  6891,   284]], device='cuda:0')

In [7]:

with torch.no_grad():
  outME = model(tokensME,output_hidden_states=True)
  outEM = model(tokensEM,output_hidden_states=True)

hs_ME = outME.hidden_states
outME.keys(), outME.hidden_states[3].shape

(odict_keys(['logits', 'past_key_values', 'hidden_states']),
 torch.Size([1, 14, 1600]))

In [8]:
len(hs_ME)

49

In [9]:
outEM.keys()

odict_keys(['logits', 'past_key_values', 'hidden_states'])

## Task_2:
- Determine whether GPT-2-xl solves the IOI task by identifying the
token with the largest logit in the final token position. Print out
both sentences with the newly generated ending.
Make a scatter plot of the raw logits in the final token posi
tion for both sentences, and highlight the vocab indices for “ Mike”
and “ Emma” as in Figure 5.51A-B. Show a scatter plot of the logits
for the final token from the two sentences as in Figure 5.51C.
Calculate the IOI score as the raw logit values for “ Mike”
minus “ Emma” in both sentences. These IOI scores will be a clean
baseline for the results of the patching experiment later in the
project